# 🎯 AI Product Recommendation Engine — IoT Edition
Recommandations logiques pour une boutique de matériel IoT/électronique.

In [ ]:
!pip install pandas numpy scikit-learn sentence-transformers -q
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.neighbors import NearestNeighbors
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
import re, unicodedata, warnings
warnings.filterwarnings("ignore")

# Le modèle sémantique (sentence-transformers) est optionnel :
# - en ligne (Colab) -> embeddings sémantiques de haute qualité
# - hors ligne        -> repli automatique sur TF-IDF (aucune installation lourde requise)
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
    print("✅ Dépendances OK — mode sémantique (sentence-transformers)")
except Exception:
    HAS_ST = False
    print("⚠️ sentence-transformers indisponible — repli automatique sur TF-IDF")

## 📊 Étape 1 : Chargement des données

In [ ]:
candidate_files = [
    Path("inventory_export.csv"),
    Path("cothings-ai/inventory_export.csv"),
    Path("data/inventory_export.csv")
]
csv_path = next((p for p in candidate_files if p.exists()), None)
if csv_path is None:
    try:
        from google.colab import files
        uploaded = files.upload()
        csv_path = Path(list(uploaded.keys())[0])
    except:
        raise FileNotFoundError("inventory_export.csv introuvable.")
df_raw = pd.read_csv(csv_path)
print(f"✅ {df_raw.shape[0]} lignes, {df_raw['Handle'].nunique()} produits uniques")
df_raw.head(3)

## 🧹 Étape 2 : Nettoyage + Catégorisation IoT

In [ ]:
df = df_raw.copy()
df.columns = df.columns.str.lower().str.strip()

def normalize_text(text):
    text = "" if pd.isna(text) else str(text)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("ascii")
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec","sur","dans","par"}
    return " ".join(t for t in text.split() if len(t)>2 and t not in stop and not t.isdigit())

# === 1) CATÉGORIES IoT (matching par MOT, pas par sous-chaîne) ===
# Important : on matche des mots entiers -> "ir" ne matche plus "solaire", "aa" ne matche plus "carameau".
CATEGORY_RULES = {
    "batterie":    ["pile","piles","batterie","battery","18650","lipo","lithium","alkaline","r20","r6","aa","aaa","4ah","accu"],
    "chargeur":    ["chargeur","charging","chargement","bouclier"],
    "led":         ["led","ruban","bande","lampe","lumiere","spectre","rgb","neon","afficheur","matrice"],
    "alimentation":["pwm","regulateur","regulator","boost","convertisseur","tension","alimentation","decoupage","7805","79l12"],
    "electronique":["module","hdmi","circuit","74hc","potentiometer","adaptateur","registre","lcd","i2c","schmitt","relais","capteur","senseur"],
    "connectique": ["cable","cables","cosses","pinces","connecteur","crocodile","jumper","cavalier","barette","fil","fils","dupont"],
    "raspberry":   ["raspberry","rpi","microbit","arduino","esp32","esp8266","nodemcu","camera"],
    "mobilite":    ["voiture","telecommandee","drone","helices","roue","chassis","4wd","robot","servomoteur"],
    "composant":   ["diode","resistance","resistances","condensateur","transistor","zener","1n4148","tube","support","quartz"],
    "outillage":   ["cutter","lame","lames","scie","outil","outils","tournevis","cle","cles","percage","foret","pistolet"],
    "rf":          ["433mhz","433","rf","stx","srx","emetteur","recepteur","bluetooth","gsm","gprs","wifi","gps","nrf24"],
}

def infer_category(title):
    words = set(normalize_text(title).split())
    for cat, kws in CATEGORY_RULES.items():
        # mot-clé court (<=3) -> doit être un mot entier ; long -> autorisé en préfixe/sous-chaîne
        for k in kws:
            if (len(k) <= 3 and k in words) or (len(k) > 3 and any(k in w for w in words)):
                return cat
    return "autre"

# === 2) EXTRACTION DE SPÉCIFICATIONS depuis le titre (le "vrai" signal qualité) ===
# Pas de prix dans les données -> on lit les specs techniques cachées dans le titre.
# Chaque motif est borné (min,max) + ancré sur \b pour ignorer les références produit (ex: DSM501A).
SPEC_PATTERNS = {
    "capacity_mah": (r'\b(\d+(?:[.,]\d+)?)\s*mah\b',  50, 100000),
    "capacity_ah":  (r'\b(\d+(?:[.,]\d+)?)\s*ah\b',  0.2,    500),
    "voltage_v":    (r'\b(\d+(?:[.,]\d+)?)\s*v\b',   0.5,    600),
    "current_ma":   (r'\b(\d+(?:[.,]\d+)?)\s*ma\b',    1, 100000),
    "current_a":    (r'\b(\d+(?:[.,]\d+)?)\s*a\b',   0.1,    200),
    "power_w":      (r'\b(\d+(?:[.,]\d+)?)\s*w\b',   0.1,   5000),
    "freq_mhz":     (r'\b(\d+(?:[.,]\d+)?)\s*m?hz\b',   1, 100000),
    "count":        (r'\b(\d+)\s*(?:pcs|pieces|broches|pin|leds?|canaux|ch)\b', 1, 10000),
}
def extract_specs(title):
    t = unicodedata.normalize("NFKD", str(title)).encode("ascii","ignore").decode("ascii").lower()
    t = t.replace("*", " ")
    out = {}
    for name, (pat, lo, hi) in SPEC_PATTERNS.items():
        vals = [float(x.replace(",",".")) for x in re.findall(pat, t)]
        vals = [v for v in vals if lo <= v <= hi]
        if vals:
            out[name] = max(vals)
    return out

# Specs où "plus grand = meilleur" -> servent à détecter un UPGRADE.
# (la tension n'en fait PAS partie : 220V n'est pas "mieux" que 12V, c'est juste différent.)
UPGRADE_SPECS = ["capacity_mah", "capacity_ah", "power_w", "count"]

def primary_spec(specs):
    """Renvoie (clé, valeur) de la spec 'qualité' principale, sinon (None, 0)."""
    for k in UPGRADE_SPECS + ["voltage_v"]:
        if k in specs:
            return k, specs[k]
    return None, 0.0

# === 3) CARTE DE COMPLÉMENTARITÉ IoT (recommandeur à base de connaissances) ===
# Sans historique d'achats, des règles métier expertes sont le bon choix (et non un défaut).
COMPLEMENTARITY_MAP = {
    "batterie":     ["chargeur", "alimentation", "composant", "connectique"],
    "chargeur":     ["batterie", "connectique", "electronique"],
    "led":          ["alimentation", "connectique", "electronique", "raspberry"],
    "alimentation": ["electronique", "connectique", "led", "batterie"],
    "electronique": ["connectique", "alimentation", "composant", "raspberry"],
    "connectique":  ["electronique", "alimentation", "composant"],
    "raspberry":    ["led", "electronique", "connectique", "alimentation", "mobilite"],
    "mobilite":     ["batterie", "chargeur", "rf", "alimentation"],
    "composant":    ["electronique", "connectique", "alimentation"],
    "outillage":    ["composant", "connectique", "electronique"],
    "rf":           ["alimentation", "electronique", "connectique", "mobilite"],
    "autre":        ["electronique", "connectique", "composant"],
}

df["product_handle"] = df["handle"].astype(str).str.strip()
df["product_title"]  = df["title"].astype(str).str.strip()
df["sku"]            = df["sku"].astype(str).str.strip() if "sku" in df.columns else ""
df["location"]       = df["location"].astype(str).str.strip() if "location" in df.columns else "unknown"

for col in ["available (not editable)","on hand (current)","on hand (new)"]:
    if col in df.columns:
        df[col] = df[col].replace("not stocked",0)
        df[col] = pd.to_numeric(df[col],errors="coerce").fillna(0).astype(int)

df["category"]    = df["product_title"].apply(infer_category)
df["clean_title"] = df["product_title"].apply(normalize_text)
df["specs"]       = df["product_title"].apply(extract_specs)

print("✅ Nettoyage + catégories + extraction de specs terminés!")
print(f"   Produits avec au moins une spec lisible : {(df.drop_duplicates('product_handle')['specs'].apply(len)>0).mean():.0%}")
print("\n📊 Distribution des catégories:")
print(df.drop_duplicates("product_handle")["category"].value_counts())

## 📈 Étape 3 : Agrégation des produits

In [ ]:
products = df.drop_duplicates(subset=["product_handle"]).copy()

inv = df.groupby("product_handle").agg({
    "available (not editable)":"sum",
    "on hand (current)":"sum",
    "on hand (new)":"sum"
}).reset_index()
inv.columns = ["product_handle","available_total","on_hand_current","on_hand_new"]

products = products.merge(inv, on="product_handle", how="left")
products["available_total"] = products["available_total"].fillna(0).astype(int)
products["product_profile"] = (
    "produit " + products["product_title"].fillna("") + " " +
    "categorie " + products["category"].fillna("") + " " +
    "motscles " + products["clean_title"].fillna("")
)
products = products[[
    "product_handle","product_title","category","sku","specs",
    "product_profile","available_total","on_hand_current","on_hand_new"
]].reset_index(drop=True)

print(f"✅ {len(products)} produits uniques")
print(f"   En stock: {len(products[products['available_total']>0])}")
products.head(5)

## 🤖 Étape 4 : Moteur de recommandation IoT

In [ ]:
class IoTRecommendationEngine:
    """
    Moteur de recommandation 'auto-pilote' pour boutique IoT/électronique.

    Pour CHAQUE produit sélectionné, il propose 3 types de recommandations,
    automatiquement, sans intervention humaine :

      ① UPGRADE      -> même produit, MEILLEURE spec (ex: batterie 2600mAh -> 3400mAh)
                        Signal de qualité extrait du TITRE (pas besoin de prix !)
      ② ALTERNATIVE  -> produit similaire / équivalent (même famille, sémantiquement proche)
      ③ GOES-WITH    -> produit complémentaire qui se prend AVEC (carte métier)

    Sources d'intelligence :
      - embeddings sémantiques (sentence-transformers) ou repli TF-IDF
      - extraction de specs techniques depuis le titre  -> axe "qualité"
      - carte de complémentarité métier                 -> axe "achète aussi"
    """

    def __init__(self, products_df, complementarity_map):
        self.products = products_df.copy().reset_index(drop=True)
        self.comp_map = complementarity_map
        self.products["clean_profile"] = self.products["product_profile"].fillna("").apply(self._normalize)

        if HAS_ST:
            print("🔄 Chargement du modèle sémantique...")
            self.model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
            texts = [f"passage: {p}" for p in self.products["clean_profile"]]
            self.X = self.model.encode(texts, normalize_embeddings=True, show_progress_bar=False)
            self.is_sparse = False
            mode = "sémantique (sentence-transformers)"
        else:
            self.vec = TfidfVectorizer(max_features=4000, ngram_range=(1,2))
            self.X = self.vec.fit_transform(self.products["clean_profile"])
            self.is_sparse = True
            mode = "TF-IDF (repli hors-ligne)"

        self.nn = NearestNeighbors(metric="cosine").fit(self.X)
        print(f"✅ Moteur prêt — {len(self.products)} produits indexés — mode {mode}")

    def _normalize(self, text):
        text = "" if pd.isna(text) else str(text)
        text = unicodedata.normalize("NFKD", text).encode("ascii","ignore").decode("ascii").lower()
        text = re.sub(r"[^a-z0-9\s]"," ",text)
        stop = {"de","des","du","la","le","les","un","une","et","au","aux","pour","avec"}
        return " ".join(t for t in text.split() if len(t)>2 and t not in stop and not t.isdigit())

    def get_product_index(self, product_title):
        if isinstance(product_title, (int, np.integer)):
            return int(product_title) if 0 <= product_title < len(self.products) else None
        q = str(product_title).lower().strip()
        m = self.products[self.products["product_title"].str.lower().str.contains(re.escape(q), na=False)]
        return int(m.index[0]) if len(m) > 0 else None

    def _query_vector(self, i):
        return self.X[i] if self.is_sparse else self.X[i:i+1]

    def _rows(self, idx_list, sim):
        """Construit un petit DataFrame lisible à partir d'une liste d'index."""
        if not idx_list:
            return pd.DataFrame(columns=["product_title","category","specs","similarity","available_total"])
        out = self.products.loc[idx_list, ["product_title","category","specs","available_total"]].copy()
        out["similarity"] = [round(sim.get(j,0.0), 3) for j in idx_list]
        return out.reset_index(drop=True)

    def recommend_smart(self, query, n=3, in_stock_only=True, sim_threshold=0.10):
        """Renvoie un dict avec les 3 buckets : upgrade / alternative / goes_with."""
        i = self.get_product_index(query)
        if i is None:
            return None
        src     = self.products.loc[i]
        src_cat = src["category"]
        pk, pv  = primary_spec(src["specs"])      # spec principale du produit source

        K = min(len(self.products), 80)
        dist, idx = self.nn.kneighbors(self._query_vector(i), n_neighbors=K)
        sim = {int(j): 1 - float(d) for d, j in zip(dist[0], idx[0])}
        order = [int(j) for j in idx[0]]

        def keep(j):
            if j == i:
                return False
            if in_stock_only and self.products.loc[j, "available_total"] <= 0:
                return False
            return sim.get(j, 0.0) >= sim_threshold

        cand = [j for j in order if keep(j)]                                  # voisins proches (substituts)
        same = [j for j in cand if self.products.loc[j, "category"] == src_cat]

        # ① UPGRADE : même catégorie, même spec "plus grand = mieux", valeur supérieure
        upgrades = []
        if pk in UPGRADE_SPECS:
            for j in same:
                jk, jv = primary_spec(self.products.loc[j, "specs"])
                if jk == pk and jv > pv:
                    upgrades.append(j)
            upgrades = sorted(
                upgrades,
                key=lambda j: (primary_spec(self.products.loc[j, "specs"])[1], sim[j]),
                reverse=True
            )[:n]

        # ② ALTERNATIVE : même catégorie, le plus proche sémantiquement (hors upgrades)
        alternatives = sorted([j for j in same if j not in upgrades],
                              key=lambda j: sim[j], reverse=True)[:n]

        # ③ GOES-WITH : catégories complémentaires (carte métier).
        # NON filtré par sim_threshold : un complément est dans une AUTRE catégorie,
        # donc sa similarité est naturellement faible -> elle sert juste à départager.
        comp_cats = self.comp_map.get(src_cat, [])
        in_comp = self.products["category"].isin(comp_cats)
        if in_stock_only:
            in_comp &= self.products["available_total"] > 0
        comp = sorted(
            [j for j in self.products.index[in_comp] if j != i],
            key=lambda j: (comp_cats.index(self.products.loc[j, "category"]),
                           -sim.get(j, 0.0),
                           -int(self.products.loc[j, "available_total"]))
        )[:n]

        return {
            "source_idx":   i,
            "source_title": src["product_title"],
            "source_cat":   src_cat,
            "primary_spec": (pk, pv),
            "upgrade":      self._rows(upgrades, sim),
            "alternative":  self._rows(alternatives, sim),
            "goes_with":    self._rows(comp, sim),
        }


engine = IoTRecommendationEngine(products, COMPLEMENTARITY_MAP)

## ⭐ Étape 5 : Test — Recommandation produit unique

In [ ]:
def spec_str(s):
    return ", ".join(f"{k}={v:g}" for k, v in s.items()) if s else "—"

def show_smart(query, n=3, in_stock_only=True):
    """Affiche les 3 recommandations auto-pilote pour un produit sélectionné."""
    res = engine.recommend_smart(query, n=n, in_stock_only=in_stock_only)
    if res is None:
        print(f"❌ Produit introuvable : '{query}'")
        return
    pk, pv = res["primary_spec"]
    print("=" * 92)
    print(f"🛒 SÉLECTION : {res['source_title']}")
    print(f"   catégorie = {res['source_cat']}   |   spec principale = {pk}={pv:g}" if pk else
          f"   catégorie = {res['source_cat']}   |   (aucune spec lisible)")
    print("=" * 92)

    blocks = [
        ("⬆️  UPGRADE  — même produit, meilleure spec", res["upgrade"]),
        ("🔁 ALTERNATIVE — équivalent / similaire",      res["alternative"]),
        ("🧩 GOES-WITH — à prendre avec (complémentaire)", res["goes_with"]),
    ]
    for title, dfb in blocks:
        print(f"\n{title}")
        if len(dfb) == 0:
            print("   (aucune recommandation)")
            continue
        for _, r in dfb.iterrows():
            print(f"   • [{r['category']:<12}] {r['product_title'][:52]:<52} "
                  f"sim={r['similarity']:.2f}  specs[{spec_str(r['specs'])}]  stock={r['available_total']}")
    print()

# Test 1 : Batterie 18650 -> doit proposer une batterie de plus grande capacité (UPGRADE)
show_smart("18650 3.7V 2600MAH")

In [ ]:
# Test 2 : Ruban LED -> ALTERNATIVE (autres rubans) + GOES-WITH (alimentation/connectique)
show_smart("Ruban LED RGB 5050")

In [ ]:
# Test 3 : Module Relais -> UPGRADE (plus de canaux / WiFi) + ALTERNATIVE (autres relais)
show_smart("Module Relais")

In [ ]:
# Test 4 : Raspberry Pi -> GOES-WITH (LED, électronique, connectique, alimentation)
show_smart("Raspberry Pi")

## 🔧 Étape 6 : Améliorer le moteur

Deux leviers, selon ce que tu veux montrer :

1. **Régler / enrichir la carte de complémentarité** (`COMPLEMENTARITY_MAP`) — c'est un *recommandeur à base de connaissances*. Sans historique d'achats, c'est le bon choix d'ingénierie (et non un défaut).
2. **Découverte automatique de familles** (clustering non supervisé) — pour montrer que le modèle peut regrouper les produits *tout seul*. ⚠️ Sur ce catalogue, le clustering reste bruité : on le garde comme **exploration**, et on s'appuie sur les catégories métier pour la production.

> 📌 **À retenir pour le coach** : « tout faire tout seul » a deux limites *de données*, pas d'IA :
> - pas de **prix/notes** → « moins cher / meilleure qualité » est remplacé par l'**upgrade de specs** (lu dans le titre) ;
> - pas d'**historique d'achats** → la complémentarité s'apprend par **règles métier**, pas par co-achat.

In [ ]:
# --- Personnaliser la carte de complémentarité (recommandeur à base de connaissances) ---
COMPLEMENTARITY_MAP["led"]      = ["alimentation", "connectique", "electronique", "raspberry"]
COMPLEMENTARITY_MAP["mobilite"] = ["batterie", "chargeur", "rf", "alimentation", "electronique"]
engine.comp_map = COMPLEMENTARITY_MAP
print("✅ Carte de complémentarité mise à jour!")
show_smart("Voiture Télécommandée")

In [ ]:
# === BONUS — Découverte AUTOMATIQUE de familles (clustering non supervisé) ===
# Montre que le modèle peut regrouper les produits *tout seul*, sans règles écrites à la main.
from sklearn.cluster import KMeans

K_FAMILIES = 25
Xc = engine.X if not engine.is_sparse else engine.X        # même représentation que le moteur
km = KMeans(n_clusters=K_FAMILIES, random_state=42, n_init=4).fit(Xc)
engine.products["auto_family"] = km.labels_

print(f"🔎 {K_FAMILIES} familles découvertes automatiquement (top mots de chaque famille) :\n")
if engine.is_sparse:
    terms = np.array(engine.vec.get_feature_names_out())
    centers = km.cluster_centers_
    for c in range(K_FAMILIES):
        top = terms[centers[c].argsort()[::-1][:6]]
        n = int((km.labels_ == c).sum())
        print(f"  famille {c:>2} (n={n:>4}) : {', '.join(top)}")
else:
    # En mode sémantique : on résume chaque famille par ses produits les plus fréquents
    for c in range(K_FAMILIES):
        members = engine.products[engine.products["auto_family"] == c]
        sample = ", ".join(members["product_title"].head(3).str[:28])
        print(f"  famille {c:>2} (n={len(members):>4}) : {sample}")

print("\n💡 Lecture pour le coach : le clustering trouve des familles seul, mais reste bruité")
print("   (catalogue large + titres hétérogènes). On garde donc les catégories métier")
print("   pour la production, et le clustering comme preuve de 'self-discovery'.")